In [1]:
!python --version  

Python 3.13.12


In [2]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

# avoid warnings for old claude model use
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# Initialize Antropic Chat client
from anthropic import Anthropic

client = Anthropic()

In [ ]:
# Helper Chat Functions


## Function to Append User messasge
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


## Function to Append Assistant messasge
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


## Main Function for chat
def get_response(messages, stop_sequences=None, system=None):

    model = "claude-haiku-4-5"  # you can change the model according to your usecase
    params = {
        "model": model,
        "max_tokens": 6000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message


def get_chat_content(message):
    return message.content[0].text

In [6]:
def getStructureOutput(mesg, data_prefix="```json", stop_sequences=None):
    messages = []

    # User's request > what user wants in specific format
    add_user_message(messages, mesg)

    # start the content and make the AI generate the remaining part.
    add_assistant_message(messages, data_prefix)

    response = get_response(messages=messages, stop_sequences=stop_sequences)
    return get_chat_content(response)

In [ ]:
import json  # For json format validation
import re  # For regex format validation
import ast  # For python syntax validation


def validate_json(data):
    try:
        json.loads(data.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(code):
    try:
        ast.parse(code.strip())
        return 10
    except:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
import json


# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    eval_text = getStructureOutput(eval_prompt, stop_sequences=["```"])
    print(eval_text)
    return json.loads(eval_text)

In [9]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    output = getStructureOutput(
        prompt,
        data_prefix="```code",  # a little cheat code to cover all the formats
        stop_sequences=["```"],
    )
    return output


def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    syntax_score = grade_syntax(output, test_case)
    score = (syntax_score + model_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "weaknesses": weaknesses,
        "strengths": strengths,
        "reasoning": reasoning,
    }


from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
import json

with open("./json_db/dataset_with_format_&_solution.json", "r") as db:
    results = run_eval(json.load(db))


{
    "strengths": [
        "Uses head_bucket() method, which is the correct and efficient AWS API for checking bucket existence",
        "Returns a boolean as required, with clear True/False logic",
        "Includes proper docstring with clear parameter and return type documentation"
    ],
    "weaknesses": [
        "Overly broad exception handling: catching all exceptions and returning False masks potential issues like credential errors, network problems, or access denied (403), making debugging difficult",
        "Does not distinguish between 404 (bucket doesn't exist) and 403 (access denied) - both return False, which could be misleading",
        "Missing error logging that would help identify actual problems vs. non-existent buckets"
    ],
    "reasoning": "The solution correctly meets the core criteria: it uses head_bucket(), attempts to handle the NoSuchBucket exception for the 404 case, and returns a boolean. However, the exception handling is problematic. The catch-al

In [12]:
# With less temprature we will likely get the almost same responses each time

# print(json.dumps(results,indent=2))